# Text-to-Outfit — Notebook 2: encoder de query (GloVe + MLP) + entrenamiento contrastivo
## (v2 — con corrección multi-positivo)

Requiere haber ejecutado antes el Notebook 1 (`text_to_outfit_01_pairing_and_embeddings.ipynb`),
que genera `t2o_samples_with_embeddings.pt`.

**Cambio respecto a la v1:** el 92,4% de los outfits comparten su texto (`title`/`url_name`) con
al menos otro outfit (ej. "Clog Sandals" aparece en 825 outfits distintos). La v1 asumía
implícitamente una correspondencia 1-a-1 texto↔outfit, lo cual generaba señal de entrenamiento
contradictoria y penalizaba injustamente en evaluación. Esta versión trata como **positivos
mutuos** a todos los outfits que comparten texto, tanto en la pérdida contrastiva como en el
cálculo de Recall@k.


In [ ]:
import json
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Preprocesado de texto — réplica exacta de `text_preprocessing.ipynb`

(Sin cambios respecto a la v1: limpieza -> tokenización NLTK -> lematización con POS ->
deduplicación de tokens. Sin el filtro `MIN_FREQ=5`, por las mismas razones que ya discutimos.)

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def tokenize(text):
    if not isinstance(text, str) or text == '':
        return []
    return word_tokenize(text)


lemmatizer = WordNetLemmatizer()


def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_map = {'J': wordnet.ADJ, 'V': wordnet.VERB, 'N': wordnet.NOUN, 'R': wordnet.ADV}
    return tag_map.get(tag, wordnet.NOUN)


def lemmatize(tokens):
    return [lemmatizer.lemmatize(t, get_wordnet_pos(t)) for t in tokens]


def remove_duplicates(tokens):
    return list(dict.fromkeys(tokens))


def preprocess_for_glove(text):
    cleaned = clean_text(text)
    tokens = tokenize(cleaned)
    tokens = lemmatize(tokens)
    tokens = remove_duplicates(tokens)
    return tokens


print(preprocess_for_glove("Wedding Guest - Fall Street Style!! 2019"))

## 2. Cargar GloVe (mismo fichero que en `text_preprocessing.ipynb`)

In [ ]:
GLOVE_PATH = "./polyvore_kaggle/polyvore_outfits/glove/wiki_giga_2024_100_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05.050_combined.txt"  # AJUSTA A TU RUTA LOCAL
EMBED_DIM = 100


def load_glove(glove_path):
    embeddings = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split()
            word = " ".join(parts[:-EMBED_DIM])
            vector = np.asarray(parts[-EMBED_DIM:], dtype=np.float32)
            embeddings[word] = vector
    return embeddings


glove = load_glove(GLOVE_PATH)
print(f"Vectores GloVe cargados: {len(glove)}")

## 3. Texto libre -> vector GloVe (100-d, L2-normalizado)

In [ ]:
def text_to_glove_vector(text, glove=glove):
    tokens = preprocess_for_glove(text)
    vecs = [glove[t] for t in tokens if t in glove]
    if not vecs:
        return np.zeros(EMBED_DIM, dtype=np.float32), True
    vec = np.mean(vecs, axis=0)
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec = vec / norm
    return vec.astype(np.float32), False

## 4. Cargar `t2o_samples`, precalcular vector de query, y asignar `group_id` por texto

**Novedad:** además de vectorizar el texto, asignamos a cada muestra un `group_id` entero --
todas las muestras con el mismo texto (dentro del mismo split) comparten `group_id`. Esto es lo
que permite tratar "Clog Sandals" (825 outfits) como 825 positivos mutuos entre sí, en vez de
como 825 negativos compitiendo por una única respuesta correcta.

In [ ]:
T2O_CACHE_PATH = "./polyvore_kaggle/polyvore_outfits/checkpoints_RE/t2o_samples_with_embeddings.pt"  # ajusta si hace falta

t2o_samples = torch.load(T2O_CACHE_PATH)
print(f"Muestras cargadas: {len(t2o_samples)}")

n_full_oov = 0
for s in tqdm(t2o_samples, desc="Vectorizando texto con GloVe"):
    vec, is_oov = text_to_glove_vector(s["text"])
    s["query_glove_vec"] = torch.from_numpy(vec)
    if is_oov:
        n_full_oov += 1

print(f"Muestras con texto totalmente OOV (vector cero): {n_full_oov}")

t2o_samples = [s for s in t2o_samples if not torch.all(s["query_glove_vec"] == 0)]
print(f"Muestras utilizables tras descartar OOV totales: {len(t2o_samples)}")

t2o_train = [s for s in t2o_samples if s["split"] == "train"]
t2o_valid = [s for s in t2o_samples if s["split"] == "valid"]
t2o_test  = [s for s in t2o_samples if s["split"] == "test"]

# cache = torch.load(T2O_CACHE_PATH)
# t2o_train, t2o_valid, t2o_test = cache["train"], cache["valid"], cache["test"]
# print(f"Cargados -> train={len(t2o_train)}  valid={len(t2o_valid)}  test={len(t2o_test)}")


# def prepare_split(samples):
#     kept = []
#     for s in samples:
#         vec, is_oov = text_to_glove_vector(s["text"])
#         if is_oov:
#             continue
#         s["query_glove_vec"] = torch.from_numpy(vec)
#         kept.append(s)
#     return kept


# t2o_train = prepare_split(t2o_train)
# t2o_valid = prepare_split(t2o_valid)
# t2o_test  = prepare_split(t2o_test)



def assign_group_ids(samples):
    """group_id local a este split: mismo texto -> mismo group_id."""
    text_to_gid = {}
    for s in samples:
        if s["text"] not in text_to_gid:
            text_to_gid[s["text"]] = len(text_to_gid)
        s["group_id"] = text_to_gid[s["text"]]
    return samples, len(text_to_gid)


t2o_train, n_groups_train = assign_group_ids(t2o_train)
t2o_valid, n_groups_valid = assign_group_ids(t2o_valid)
t2o_test,  n_groups_test  = assign_group_ids(t2o_test)

print(f"train={len(t2o_train)} ({n_groups_train} textos únicos)  "
      f"valid={len(t2o_valid)} ({n_groups_valid} textos únicos)  "
      f"test={len(t2o_test)} ({n_groups_test} textos únicos)")

## 5. Dataset + DataLoader

Ahora cada muestra devuelve también su `group_id`, necesario para construir la máscara de
positivos múltiples dentro del batch.

In [ ]:
class T2OQueryDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return s["outfit_emb"], s["query_glove_vec"], s["group_id"]


def make_t2o_loader(samples, batch_size=128, shuffle=True):
    ds = T2OQueryDataset(samples)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)


BATCH_SIZE = 128

t2o_train_loader = make_t2o_loader(t2o_train, batch_size=BATCH_SIZE, shuffle=True)
t2o_valid_loader = make_t2o_loader(t2o_valid, batch_size=BATCH_SIZE, shuffle=False)
t2o_test_loader  = make_t2o_loader(t2o_test,  batch_size=BATCH_SIZE, shuffle=False)

## 6. Encoder de query: GloVe (100-d) -> MLP -> 128-d

(Sin cambios respecto a la v1.)

In [ ]:
class QueryEncoder(nn.Module):
    def __init__(self, input_dim=EMBED_DIM, hidden_dim=256, output_dim=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, glove_vec):
        x = self.net(glove_vec)
        return F.normalize(x, p=2, dim=-1)


query_encoder = QueryEncoder().to(device)
n_params = sum(p.numel() for p in query_encoder.parameters())
print(f"Parámetros del query encoder: {n_params:,}")

## 7. Pérdida contrastiva multi-positivo (InfoNCE enmascarada, estilo SupCon)

**Este es el cambio clave.** En vez de `labels = torch.arange(B)` (que asume "la única respuesta
correcta de la fila i es la columna i"), construimos una máscara `positive_mask` de tamaño
`(B, B)`: `True` en la posición `(i, j)` si las muestras `i` y `j` del batch comparten texto
(mismo `group_id`). La pérdida de cada fila se promedia sobre **todos** sus positivos (puede
haber más de uno en el batch), no solo sobre la diagonal. Si ninguna otra muestra del batch
comparte texto con `i`, se reduce automáticamente al caso normal de un único positivo (la
diagonal), así que esta versión generaliza a la anterior sin penalizar los casos ya correctos.

In [ ]:
class ContrastiveHead(nn.Module):
    def __init__(self, init_temperature=0.07):
        super().__init__()
        self.log_temp = nn.Parameter(torch.tensor(np.log(1.0 / init_temperature)))

    def forward(self):
        return self.log_temp.exp().clamp(max=100.0)


contrastive_head = ContrastiveHead().to(device)


def masked_nce(logits, positive_mask):
    """logits: (B, B) similitudes ya escaladas por temperatura.
    positive_mask: (B, B) bool/float, True donde la columna es un positivo válido para esa fila."""
    log_prob = F.log_softmax(logits, dim=1)
    pos_count = positive_mask.sum(dim=1).clamp(min=1.0)
    loss_per_row = -(log_prob * positive_mask).sum(dim=1) / pos_count
    return loss_per_row.mean()


def contrastive_loss(outfit_emb, query_emb, group_ids, temperature):
    """outfit_emb, query_emb: (B, 128) L2-normalizados. group_ids: (B,) enteros --
    mismo group_id = mismo texto = positivos mutuos (no se fuerzan como negativos)."""
    logits = outfit_emb @ query_emb.T * temperature          # (B, B)
    positive_mask = (group_ids.unsqueeze(1) == group_ids.unsqueeze(0)).float()  # (B, B)

    loss_o2q = masked_nce(logits, positive_mask)
    loss_q2o = masked_nce(logits.T, positive_mask.T)  # positive_mask es simétrica, pero por claridad
    return (loss_o2q + loss_q2o) / 2

## 8. Métrica de evaluación: Recall@k multi-positivo sobre el corpus de test completo

**También cambia aquí.** Para cada query, en vez de mirar solo el rank del outfit con el mismo
índice, buscamos el **mejor rank entre todos los outfits del split que comparten su texto**
(su `group_id`). Si el modelo recupera cualquier outfit válido de ese grupo dentro del top-k,
cuenta como acierto.

In [ ]:
@torch.no_grad()
def evaluate_recall_at_k(samples, ks=(1, 5, 10, 30, 50)):
    query_encoder.eval()
    outfit_embs = torch.stack([s["outfit_emb"] for s in samples]).to(device)        # (N, 128)
    glove_vecs  = torch.stack([s["query_glove_vec"] for s in samples]).to(device)   # (N, 100)
    group_ids   = torch.tensor([s["group_id"] for s in samples], device=device)     # (N,)

    query_embs = query_encoder(glove_vecs)             # (N, 128)
    sims = query_embs @ outfit_embs.T                  # (N, N) -- fila=query, columna=outfit candidato

    n = sims.shape[0]
    relevant_mask = (group_ids.unsqueeze(1) == group_ids.unsqueeze(0))  # (N, N) True si mismo texto

    sorted_idx = sims.argsort(dim=1, descending=True)                   # (N, N)
    ranks_matrix = torch.empty_like(sorted_idx)
    rank_values = torch.arange(1, n + 1, device=device).unsqueeze(0).expand(n, -1)
    ranks_matrix.scatter_(1, sorted_idx, rank_values)                   # rank (1-indexed) de cada columna

    # de todos los outfits relevantes de la fila, nos quedamos con el de MEJOR (menor) rank
    ranks_masked = torch.where(relevant_mask, ranks_matrix, torch.full_like(ranks_matrix, n + 1))
    best_rank = ranks_masked.min(dim=1).values.float()                  # (N,)

    results = {f"Recall@{k}": (best_rank <= k).float().mean().item() for k in ks}
    results["MedianRank"] = best_rank.median().item()
    query_encoder.train()
    return results

## 9. Bucle de entrenamiento (early stopping sobre Recall@5 en validación)

In [ ]:
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4
EPOCHS        = 100
PATIENCE      = 12

optimizer = torch.optim.AdamW(
    list(query_encoder.parameters()) + list(contrastive_head.parameters()),
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
)

CKPT_PATH_QUERY_ENCODER = "./polyvore_kaggle/polyvore_outfits/checkpoints_RE/best_query_encoder_zero_shot.pt"

best_recall5 = -1.0
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    query_encoder.train()
    train_losses = []
    for outfit_emb, glove_vec, group_id in t2o_train_loader:
        outfit_emb = outfit_emb.to(device)
        glove_vec  = glove_vec.to(device)
        group_id   = group_id.to(device)

        query_emb = query_encoder(glove_vec)
        temperature = contrastive_head()
        loss = contrastive_loss(outfit_emb, query_emb, group_id, temperature)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    val_metrics = evaluate_recall_at_k(t2o_valid)
    train_loss_mean = np.mean(train_losses)

    print(f"Época {epoch:3d} | loss={train_loss_mean:.4f} | "
          f"val R@1={val_metrics['Recall@1']:.3f} R@5={val_metrics['Recall@5']:.3f} "
          f"R@10={val_metrics['Recall@10']:.3f} medRank={val_metrics['MedianRank']:.1f}")

    if val_metrics["Recall@5"] > best_recall5:
        best_recall5 = val_metrics["Recall@5"]
        epochs_no_improve = 0
        torch.save({
            "epoch": epoch,
            "query_encoder_state_dict": query_encoder.state_dict(),
            "contrastive_head_state_dict": contrastive_head.state_dict(),
            "val_recall5": best_recall5,
        }, CKPT_PATH_QUERY_ENCODER)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping en época {epoch} (sin mejora en {PATIENCE} épocas)")
            break

print(f"\nMejor Recall@5 en validación: {best_recall5:.3f}")

## 10. Evaluación final sobre test (con el mejor checkpoint)

In [ ]:
best_ckpt = torch.load(CKPT_PATH_QUERY_ENCODER, map_location=device)
query_encoder.load_state_dict(best_ckpt["query_encoder_state_dict"])
contrastive_head.load_state_dict(best_ckpt["contrastive_head_state_dict"])

test_metrics = evaluate_recall_at_k(t2o_test)
print("Resultados en test (multi-positivo):")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.3f}")

## 11. Inferencia: recuperar outfits a partir de una consulta libre

(Sin cambios respecto a la v1 -- la inferencia real no necesita `group_id`, eso solo importa
para entrenamiento/evaluación donde comparamos contra respuestas conocidas.)

In [ ]:
@torch.no_grad()
def retrieve_outfits(query_text, candidate_samples=t2o_test, top_k=5):
    query_encoder.eval()
    vec, is_oov = text_to_glove_vector(query_text)
    if is_oov:
        print("Aviso: la consulta no tiene ninguna palabra reconocida por GloVe.")
    glove_vec = torch.from_numpy(vec).unsqueeze(0).to(device)
    query_emb = query_encoder(glove_vec)

    outfit_embs = torch.stack([s["outfit_emb"] for s in candidate_samples]).to(device)
    sims = (query_emb @ outfit_embs.T).squeeze(0)

    top_scores, top_idx = sims.topk(top_k)
    results = []
    for score, idx in zip(top_scores.tolist(), top_idx.tolist()):
        s = candidate_samples[idx]
        results.append({"set_id": s["set_id"], "text": s["text"], "score": score})
    return results


for r in retrieve_outfits("formal office outfit neutral tones"):
    score = r["score"]
    set_id = r["set_id"]
    text = r["text"]
    print(f"{score:.3f}  {set_id:>10}  {text}")


---
### Siguiente paso

Con esto corregido, el resultado ya no estará penalizado por la duplicidad de textos. El
siguiente paso pendiente son las Capas 1 (filtro de spam/afiliación) y 4 (cap de frecuencia por
etiqueta), que se añadirán sobre este mismo notebook en la Celda 4 (donde se construyen
`t2o_train`/`t2o_valid`/`t2o_test`).